# Análisis de sentimiento tradicional en español

## Contexto práctico

Una empresa recibe comentarios de clientes después de una compra o una atención. El equipo quiere saber si las opiniones expresan satisfacción, molestia o simplemente una solicitud informativa.

En este notebook construiremos un analizador de sentimiento tradicional basado en un **diccionario de palabras positivas y negativas**. El método es transparente: podremos observar qué palabras aportaron puntos al resultado.

El objetivo no es comprender toda la intención humana con perfección. Es obtener una primera medición sencilla y explicable de la percepción de los clientes.

## ¿Qué problema se resuelve?

Leer manualmente miles de comentarios puede ser lento. Un análisis de sentimiento ayuda a detectar rápidamente:

- si predominan opiniones positivas o negativas;
- qué comentarios deberían revisarse primero;
- si un canal o servicio genera más insatisfacción;
- si una mejora produjo cambios en la percepción.

La salida será una clasificación y un puntaje. La decisión final todavía requiere contexto, revisión humana y datos adicionales.

## Objetivos de aprendizaje

- Preparar comentarios en español.
- Construir un diccionario de sentimiento.
- Calcular un puntaje por comentario.
- Considerar negaciones sencillas como “no funciona”.
- Clasificar opiniones como positivas, negativas o neutrales.
- Comparar el resultado con etiquetas de referencia.
- Interpretar gráficas y errores.
- Traducir el análisis a acciones de negocio.

Los datos están incluidos dentro del notebook y no se necesita subir un archivo externo.

## ¿Cómo funciona el método tradicional?

El método sigue una regla sencilla:

1. localizar palabras positivas y negativas;
2. sumar puntos positivos;
3. restar puntos negativos;
4. invertir el significado cuando aparece una negación cercana;
5. convertir el puntaje final en positivo, negativo o neutral.

Ejemplo: “el servicio fue excelente” suma puntos. “el servicio no fue excelente” cambia el efecto de excelente y reduce la puntuación.

## 1. Preparar el entorno

Usaremos pandas para organizar los datos, scikit-learn para las métricas de evaluación y Plotly para crear gráficas interactivas.

In [ ]:
%%capture
!pip -q install pandas scikit-learn plotly

In [ ]:
import re
import unicodedata
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option('display.max_colwidth', 140)
print('Entorno preparado')

## 2. Dataset incluido en el notebook

La columna sentimiento_real representa una etiqueta de referencia preparada para este ejercicio. El analizador no la utilizará para calcular el puntaje; se usará después para revisar qué tan bien funcionó la regla.

In [ ]:
datos = [
    {'id': 'M001', 'canal': 'encuesta', 'comentario': 'La atención fue excelente y resolvieron mi problema rápidamente.', 'sentimiento_real': 'positivo'},
    {'id': 'M002', 'canal': 'chat', 'comentario': 'Estoy muy satisfecho con la compra y recomiendo el servicio.', 'sentimiento_real': 'positivo'},
    {'id': 'M003', 'canal': 'correo', 'comentario': 'El producto llegó perfecto y la entrega cumplió la fecha.', 'sentimiento_real': 'positivo'},
    {'id': 'M004', 'canal': 'encuesta', 'comentario': 'La respuesta fue amable, clara y eficiente.', 'sentimiento_real': 'positivo'},
    {'id': 'M005', 'canal': 'chat', 'comentario': 'Estoy contento porque resolvieron el caso sin complicaciones.', 'sentimiento_real': 'positivo'},
    {'id': 'M006', 'canal': 'correo', 'comentario': 'La experiencia fue buena y volvería a comprar.', 'sentimiento_real': 'positivo'},
    {'id': 'M007', 'canal': 'encuesta', 'comentario': 'El pedido llegó tarde y hubo retraso en la entrega.', 'sentimiento_real': 'negativo'},
    {'id': 'M008', 'canal': 'chat', 'comentario': 'Estoy molesto porque el producto llegó dañado.', 'sentimiento_real': 'negativo'},
    {'id': 'M009', 'canal': 'correo', 'comentario': 'La respuesta fue lenta y dejó mi caso pendiente.', 'sentimiento_real': 'negativo'},
    {'id': 'M010', 'canal': 'encuesta', 'comentario': 'El servicio fue pésimo y tuve muchos errores en la factura.', 'sentimiento_real': 'negativo'},
    {'id': 'M011', 'canal': 'chat', 'comentario': 'Estoy decepcionado, la solución fue incompleta.', 'sentimiento_real': 'negativo'},
    {'id': 'M012', 'canal': 'correo', 'comentario': 'La experiencia fue mala y el problema continúa.', 'sentimiento_real': 'negativo'},
    {'id': 'M013', 'canal': 'encuesta', 'comentario': 'Solicito información sobre el horario de atención.', 'sentimiento_real': 'neutral'},
    {'id': 'M014', 'canal': 'chat', 'comentario': 'Quisiera conocer el estado de mi pedido.', 'sentimiento_real': 'neutral'},
    {'id': 'M015', 'canal': 'correo', 'comentario': 'Necesito actualizar los datos de mi cuenta.', 'sentimiento_real': 'neutral'},
    {'id': 'M016', 'canal': 'encuesta', 'comentario': '¿Cuál es el costo del envío a mi ciudad?', 'sentimiento_real': 'neutral'},
    {'id': 'M017', 'canal': 'chat', 'comentario': 'No fue excelente, pero la solución fue oportuna.', 'sentimiento_real': 'neutral'},
    {'id': 'M018', 'canal': 'correo', 'comentario': 'El producto funciona, aunque la entrega fue tarde.', 'sentimiento_real': 'neutral'},
    {'id': 'M019', 'canal': 'encuesta', 'comentario': 'La aplicación no funciona correctamente.', 'sentimiento_real': 'negativo'},
    {'id': 'M020', 'canal': 'chat', 'comentario': 'No estoy satisfecho con la respuesta recibida.', 'sentimiento_real': 'negativo'},
    {'id': 'M021', 'canal': 'correo', 'comentario': 'Agradezco la respuesta y la solución fue fácil.', 'sentimiento_real': 'positivo'},
    {'id': 'M022', 'canal': 'encuesta', 'comentario': 'El proceso fue claro, pero quedó un pendiente.', 'sentimiento_real': 'neutral'},
    {'id': 'M023', 'canal': 'chat', 'comentario': 'La compra fue buena y el servicio amable.', 'sentimiento_real': 'positivo'},
    {'id': 'M024', 'canal': 'correo', 'comentario': 'El cobro fue duplicado y necesito una aclaración.', 'sentimiento_real': 'negativo'},
]

df = pd.DataFrame(datos)
print(f'Dataset cargado correctamente: {len(df)} comentarios y {df.shape[1]} columnas.')
display(df.head())


### Interpretación de los datos

La etiqueta real sirve como referencia para comprobar el método, pero no representa una verdad absoluta. En comentarios reales puede haber ironía, opiniones mixtas, frases ambiguas o diferencias entre anotadores.

In [ ]:
distribucion = df['sentimiento_real'].value_counts().rename_axis('sentimiento').reset_index(name='cantidad')
display(distribucion)
fig = px.bar(distribucion, x='sentimiento', y='cantidad', text='cantidad', color='sentimiento', color_discrete_map={'positivo': '#2E8B57', 'negativo': '#C0504D', 'neutral': '#7F8C8D'})
fig.update_traces(textposition='outside')
fig.update_layout(title='Distribución de etiquetas de referencia', yaxis_title='Cantidad de comentarios', xaxis_title='', showlegend=False, height=400)
fig.show()

## 3. Normalizar los comentarios

La función convierte a minúsculas, elimina acentos y conserva solo palabras y espacios. Esto permite que “rápidamente” y “rapidamente” se comparen de forma similar.

La normalización no decide el sentimiento; solamente prepara el texto para buscar palabras en el diccionario.

In [ ]:
def normalizar_texto(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

df['texto_normalizado'] = df['comentario'].apply(normalizar_texto)
display(df[['comentario', 'texto_normalizado']].head(8))

## 4. Crear el diccionario de sentimiento

Cada palabra positiva suma un punto y cada palabra negativa resta un punto. Este diccionario es pequeño para que el funcionamiento sea visible y modificable.

En un proyecto real se ampliaría con vocabulario del dominio, variantes gramaticales y expresiones de varias palabras.

In [ ]:
palabras_positivas = {
    'excelente', 'rapidamente', 'satisfecho', 'recomiendo', 'perfecto', 'funciona',
    'cumplio', 'amable', 'clara', 'eficiente', 'contento', 'buena',
    'volveria', 'agradezco', 'solucion', 'oportuna', 'bien', 'facil'
}

palabras_negativas = {
    'tarde', 'retraso', 'molesto', 'errores', 'error',
    'decepcionado', 'danado', 'pesimo', 'grave', 'lenta', 'pendiente',
    'mala', 'incompleta', 'problema', 'duplicado'
}

negadores = {'no', 'nunca', 'jamas', 'sin'}
print(f'Palabras positivas: {len(palabras_positivas)}')
print(f'Palabras negativas: {len(palabras_negativas)}')

### ¿Por qué necesitamos negaciones?

La palabra “excelente” normalmente es positiva, pero “no excelente” cambia la interpretación. La regla siguiente revisa las tres palabras anteriores a cada término de sentimiento y, si encuentra un negador, invierte su valor.

Es una regla sencilla. No resuelve ironía ni negaciones largas, pero hace visible una mejora importante sobre contar palabras sin contexto.

## 5. Calcular puntaje y clasificación

La función devuelve el puntaje, la clasificación y las palabras que justificaron el resultado. Esto permite explicar el modelo, no solo producir una etiqueta.

In [ ]:
def analizar_sentimiento(texto):
    tokens = normalizar_texto(texto).split()
    puntaje = 0
    evidencia = []

    for posicion, token in enumerate(tokens):
        if token in palabras_positivas or token in palabras_negativas:
            valor = 1 if token in palabras_positivas else -1
            ventana = tokens[max(0, posicion - 3):posicion]
            invertido = any(palabra in negadores for palabra in ventana)
            if invertido:
                valor *= -1
            puntaje += valor
            evidencia.append({'palabra': token, 'aporte': valor, 'negacion_cercana': invertido})

    if puntaje > 0:
        etiqueta = 'positivo'
    elif puntaje < 0:
        etiqueta = 'negativo'
    else:
        etiqueta = 'neutral'
    return puntaje, etiqueta, evidencia

In [ ]:
ejemplos = [
    'La atención fue excelente y muy clara.',
    'El envío fue tarde y el retraso fue molesto.',
    'No fue excelente, pero la solución fue oportuna.',
    'Solicito información sobre mi factura.'
]

for ejemplo in ejemplos:
    puntaje, etiqueta, evidencia = analizar_sentimiento(ejemplo)
    print(f'Texto: {ejemplo}')
    print(f'Puntaje: {puntaje} | Clasificación: {etiqueta} | Evidencia: {evidencia}\n')

### Interpretación de los ejemplos

La evidencia muestra qué palabras contribuyeron al resultado. Esto es útil para explicar por qué el método clasificó una opinión de cierta manera y para detectar palabras que deberían entrar o salir del diccionario.

## 6. Analizar todos los comentarios

Aplicaremos la función a cada registro y guardaremos el puntaje, la etiqueta predicha y las palabras utilizadas como evidencia.

In [ ]:
resultados = df['comentario'].apply(analizar_sentimiento)
df['puntaje'] = resultados.apply(lambda resultado: resultado[0])
df['sentimiento_predicho'] = resultados.apply(lambda resultado: resultado[1])
df['evidencia'] = resultados.apply(lambda resultado: ', '.join(item['palabra'] for item in resultado[2]))
display(df[['id', 'comentario', 'puntaje', 'sentimiento_predicho', 'evidencia']])

### Interpretación del puntaje

El puntaje no es una probabilidad. Es un balance simple entre palabras positivas y negativas. Un puntaje de 2 significa que hubo dos puntos netos a favor; no significa 80% de confianza.

Los comentarios con puntaje cero pueden ser neutrales o pueden contener opiniones mezcladas que el diccionario no logró distinguir.

In [ ]:
comparacion = pd.crosstab(df['sentimiento_real'], df['sentimiento_predicho'])
print(f'Exactitud: {accuracy_score(df.sentimiento_real, df.sentimiento_predicho):.2%}')
print(classification_report(df.sentimiento_real, df.sentimiento_predicho, zero_division=0))
display(comparacion)

### ¿Cómo leer la evaluación?

La exactitud indica la proporción total de comentarios acertados. Precision, recall y F1 permiten revisar cada sentimiento por separado.

En un proyecto real, un falso negativo —no detectar una opinión negativa— podría ser más costoso que revisar manualmente una opinión neutral. Por eso la métrica prioritaria debe definirse con el área de negocio.

In [ ]:
etiquetas = ['positivo', 'neutral', 'negativo']
matriz = confusion_matrix(df['sentimiento_real'], df['sentimiento_predicho'], labels=etiquetas)
fig = px.imshow(matriz, x=etiquetas, y=etiquetas, text_auto=True, color_continuous_scale='Blues', zmin=0, labels={'x': 'Sentimiento predicho', 'y': 'Sentimiento real', 'color': 'Cantidad'})
fig.update_layout(title='Matriz de confusión del análisis de sentimiento', height=500)
fig.show()

### Interpretación de la matriz de confusión

Las celdas de la diagonal representan aciertos. Las celdas fuera de la diagonal representan confusiones.

Por ejemplo, si un comentario real negativo aparece como neutral, el diccionario no encontró suficiente vocabulario negativo. Si aparece como positivo, probablemente existen palabras positivas y negativas mezcladas o falta una regla de contexto.

## 7. Visualizar la percepción por canal

La siguiente gráfica ayuda a identificar si un canal concentra más opiniones negativas. La cantidad de comentarios también debe observarse: una proporción alta basada en muy pocos registros puede ser inestable.

In [ ]:
por_canal = df.groupby(['canal', 'sentimiento_predicho']).size().reset_index(name='cantidad')
fig = px.bar(por_canal, x='canal', y='cantidad', color='sentimiento_predicho', barmode='group', text='cantidad', color_discrete_map={'positivo': '#2E8B57', 'negativo': '#C0504D', 'neutral': '#7F8C8D'}, labels={'canal': 'Canal', 'cantidad': 'Comentarios', 'sentimiento_predicho': 'Sentimiento'})
fig.update_traces(textposition='outside')
fig.update_layout(title='Sentimiento predicho por canal', height=500)
fig.show()

## 8. Probar comentarios nuevos

Este bloque simula el uso diario del analizador. La evidencia permite que el usuario revise por qué se asignó la etiqueta.

In [ ]:
nuevos_comentarios = [
    'La solución fue rápida y estoy muy satisfecho.',
    'El pedido no llegó y el servicio fue terrible.',
    'Necesito saber cómo actualizar mi dirección.'
]

nuevos_resultados = []
for texto in nuevos_comentarios:
    puntaje, etiqueta, evidencia = analizar_sentimiento(texto)
    nuevos_resultados.append({
        'comentario': texto,
        'puntaje': puntaje,
        'sentimiento': etiqueta,
        'evidencia': ', '.join(item['palabra'] for item in evidencia)
    })

display(pd.DataFrame(nuevos_resultados))

## 9. Conclusiones generales

1. El análisis tradicional basado en diccionario es fácil de explicar y auditar.
2. El puntaje muestra un balance de palabras, no una probabilidad de sentimiento.
3. Las negaciones sencillas mejoran la interpretación, pero no resuelven ironía ni contexto complejo.
4. La evaluación debe revisar errores por sentimiento, no solo exactitud total.
5. La evidencia textual ayuda a ampliar el diccionario y corregir reglas.
6. Las gráficas permiten detectar canales o áreas con mayor concentración de opiniones negativas.
7. Antes de producción se deben proteger datos personales, ampliar el vocabulario y validar con comentarios reales.

### Conclusión ejecutiva

Este método es una buena primera aproximación cuando se necesita una medición rápida, económica y transparente. Puede apoyar el monitoreo de satisfacción y la priorización de casos, pero no debe reemplazar la revisión humana en comentarios ambiguos o decisiones importantes.